In [1]:
import re, json, datetime as dt
from collections import defaultdict, deque
from pathlib import Path
import pandas as pd
import altair as alt

import theme

alt.themes.register('main_theme', theme.main_theme)
alt.themes.enable('main_theme')

ThemeRegistry.enable('main_theme')

In [22]:
TREE_FILE = Path("data/nextstrain_h3n2_ha_12y/h3n2_ha_12y_nextstrain.json")
FREQ_FILE  = Path("data/nextstrain_h3n2_ha_12y/h3n2_ha_12y_frequencies.json")

SITES_OF_INTEREST = {159, 160, 195}

mut_pat = re.compile(r'([A-Z\-])(\d+)([A-Z\*\-])') # e.g. Y159N

# Nextstrain decimal-year → datetime
def decyear_to_date(y):
    year, rem = int(y), y % 1
    return dt.datetime(year, 1, 1) + dt.timedelta(days=rem*365.25)

# Load tree and accumulate all mutations on every node
with TREE_FILE.open() as f:
    tree_json = json.load(f)

tree = tree_json['tree']
ROOT_HA1 = tree_json['root_sequence']['HA1']
ROOT_HA2 = tree_json['root_sequence']['HA2']

# Get the root residue is at each site
ROOT_ALLELES = {
    159: ROOT_HA1[159-1], 
    160: ROOT_HA1[160-1], 
    195: ROOT_HA1[195-1]
}

print(ROOT_ALLELES)

def cumulative_mutations(root):
    out, stack = {}, deque([(root, defaultdict(list))])
    while stack:
        node, parent_muts = stack.pop()
        muts = {g: list(m) for g, m in parent_muts.items()}
        for gene, mlist in node.get("branch_attrs", {}).get("mutations", {}).items():
            muts.setdefault(gene, []).extend(mlist)
        out[node["name"]] = muts
        for child in node.get("children", []):
            stack.append((child, muts))
    return out

cum_muts = cumulative_mutations(tree)

def find_tips(n):
    if not n.get("children"):
        return [n["name"]]
    tips = []
    for c in n["children"]:
        tips.extend(find_tips(c))
    return tips

tips = set(find_tips(tree))

# Load frequencies
with FREQ_FILE.open() as f:
    freq_json = json.load(f)

pivots = freq_json["pivots"]
dates  = [decyear_to_date(x) for x in pivots]

# Build a *complete* label: like 159-160-195
def haplotype_label(muts):
    # start with the background residues
    site_allele = dict(ROOT_ALLELES)

    # overwrite with any mutations that hit those sites
    for mut in muts.get("HA1", []):
        m = mut_pat.match(mut)
        if not m: # unexpected format
            continue
        _, pos_str, new_aa = m.groups()
        pos = int(pos_str)
        if pos in SITES_OF_INTEREST:
            site_allele[pos] = new_aa

    # guaranteed to have *all* sites
    return "-".join(f"{pos}{site_allele[pos]}" for pos in sorted(SITES_OF_INTEREST))

# Summarize frequencies
hap_freqs = defaultdict(lambda: [0.0] * len(pivots))
for tip in tips:
    label = haplotype_label(cum_muts[tip])
    for i, f in enumerate(freq_json[tip]["frequencies"]):
        hap_freqs[label][i] += f

records = [
    {
        "date"     : dates[i],
        "pivot"    : pivots[i],
        "haplotype": label,
        "frequency": freq,
    }
    for label, series in hap_freqs.items()
    for i, freq in enumerate(series)
]

df = pd.DataFrame(records)

{159: 'F', 160: 'K', 195: 'Y'}


In [23]:
haplos = sorted(df['haplotype'].unique())
ggsci_cat20 = ["#393B79", "#637939", "#8C6D31", "#843C39", "#7B4173", "#5254A3", "#8CA252", "#BD9E39", 
                "#AD494A", "#A55194", "#6B6ECF", "#B5CF6B", "#E7BA52", "#D6616B", "#CE6DBD", "#9C9EDE", 
                "#CEDB9C", "#E7CB94", "#E7969C", "#DE9ED6"]
colors = ggsci_cat20[::-1][:len(haplos)]

alt.Chart(df.query('date > 2018')).mark_area(stroke='black', strokeWidth=0.5, opacity=0.8).encode(
    x=alt.X("date:T", title="Date"),
    y=alt.Y("frequency:Q", title="Frequency").stack("normalize"),
    color=alt.Color(
        "haplotype:N", 
        scale=alt.Scale(range=colors, domain=haplos), 
        legend=alt.Legend(title="Haplotype")
    ),
    order=alt.Order("haplotype:N"),
    tooltip=['date', 'haplotype', 'frequency']
).properties(
    width=400,
    height=200,
)

alt.Chart(...)

In [25]:
haplos_of_interest = df.query('date > 2018 and frequency >= 0.2')['haplotype'].unique()

df = df.assign(
    background=lambda d: d['haplotype'].where(d['haplotype'].isin(haplos_of_interest), 'Other')
)
df.head()

,date,pivot,haplotype,frequency,background
0,2013-04-24 14:13:33.599998,2013.3110,159Y-160T-195Y,0.0,159Y-160T-195Y
1,2013-05-24 14:47:28.319999,2013.3932,159Y-160T-195Y,0.0,159Y-160T-195Y
2,2013-06-24 15:01:28.560001,2013.4781,159Y-160T-195Y,0.0,159Y-160T-195Y
3,2013-07-24 15:35:23.280003,2013.5603,159Y-160T-195Y,0.0,159Y-160T-195Y
4,2013-08-24 15:49:23.519998,2013.6452,159Y-160T-195Y,0.0,159Y-160T-195Y


In [33]:
haplos = ['159Y-160T-195Y', '159Y-160K-195Y', '159S-160K-195Y', '159Y-160T-195F', '159N-160I-195F', 'Other']
colors = ['#3969AC', '#7F3C8D', '#E73F74', '#80BA5A', '#F2B701', '#CBCACB']

alt.Chart(df.query('date > 2018')).mark_area(stroke='black', strokeWidth=0.5, opacity=0.7).encode(
    x=alt.X("date:T", title="Date"),
    y=alt.Y("frequency:Q", title="Frequency").stack("normalize"),
    color=alt.Color(
        "background:N", 
        scale=alt.Scale(range=colors, domain=haplos), 
        legend=alt.Legend(title="Haplotype")
    ),
    order=alt.Order("haplotype:N"),
    tooltip=['date', 'haplotype', 'background', 'frequency']
).properties(
    width=400,
    height=200,
)

alt.Chart(...)